# Final Results: Full Bias Analysis — LLaMA-2 vs PersianLLaMA

This is the **master analysis notebook** that:
1. Loads all precomputed scores (PLL, CB, APX)
2. Compares LLaMA-2 and PersianLLaMA across all bias types
3. Generates plots and summary tables
4. Analyzes tokenizer fragmentation effects

**Link to original Colab**: https://colab.research.google.com/drive/1g5b5Mptu12Of0lzMf4x_bp1dbSclwoas

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.2)

## 1. Load All Result Files

In [ ]:
BASE = '/content/drive/MyDrive/'

# PLL results
BiasPe = pd.read_csv(BASE + 'stereoset_persianLLama.csv')
BiasPe['model'] = 'PersianLLaMA'
Biasen = pd.read_csv(BASE + 'stereoset_LLama.csv')
Biasen['model'] = 'LLaMA'

# CB results by bias type
BiasbyPe = pd.read_csv(BASE + 'stereoset_persianLLama_ByBiasType.csv')
BiasbyEn = pd.read_csv(BASE + 'stereoset_LLama_ByBiasType.csv')

# APX results
apx_llama = pd.read_csv(BASE + 'APX_LLaMA.csv')
apx_persian = pd.read_csv(BASE + 'APX_persianLLaMA.csv')

# Tokenizer features
tok_df = pd.read_csv(BASE + 'stereoset_with_tokenizer_features.csv')

print('All files loaded.')

## 2. PLL Bias Score Comparison

In [ ]:
df = pd.concat([Biasen, BiasPe], ignore_index=True)
df = df[df['labelname'].isin(['stereotype', 'anti-stereotype', 'unrelated'])]
df['biaslabel'] = df['biastype'] + ' - ' + df['labelname']

# BiasScore summary
print('=== Mean BiasScore (PLL) by model and bias type ===')
for model_name, mgroup in df.groupby('model'):
 print(f'\n{model_name}:')
 col = 'BiasEnglish' if model_name == 'LLaMA' else 'BiasPersian'
 pll_summary = mgroup[mgroup['labelname']=='stereotype'].groupby('biastype')[col].mean()
 print(pll_summary.round(3))

## 3. CB Score Comparison by Bias Type

In [ ]:
cb_llama = BiasbyEn.groupby('biastype')['CBEnglish'].mean().rename('LLaMA')
cb_persian = BiasbyPe.groupby('biastype')['CBPersian'].mean().rename('PersianLLaMA')
cb_compare = pd.concat([cb_llama, cb_persian], axis=1).round(3)
print('=== CB by Bias Type ===')
print(cb_compare)

In [ ]:
# CB Bar Plot
cb_compare.plot(kind='bar', figsize=(10, 6), color=['steelblue', 'coral'])
plt.title('Categorical Bias (CB) by Bias Type: LLaMA vs PersianLLaMA')
plt.ylabel('CB Score')
plt.xlabel('Bias Type')
plt.xticks(rotation=0)
plt.legend(title='Model')
plt.tight_layout()
plt.savefig('cb_comparison.png', dpi=150)
plt.show()

## 4. APX Score Comparison

In [ ]:
print('=== APX Bias by Type (LLaMA-2) ===')
for btype, group in apx_llama.groupby('bias_type'):
 s = group[group['label_name']=='stereotype']['APX_English'].mean()
 a = group[group['label_name']=='anti-stereotype']['APX_English'].mean()
 print(f'{btype}: stereotype={s:.2f}, anti-stereotype={a:.2f}, diff={s-a:.2f}')

print('\n=== APX Bias by Type (PersianLLaMA) ===')
for btype, group in apx_persian.groupby('bias_type'):
 s = group[group['label_name']=='stereotype']['APX_Persian'].mean()
 a = group[group['label_name']=='anti-stereotype']['APX_Persian'].mean()
 print(f'{btype}: stereotype={s:.2f}, anti-stereotype={a:.2f}, diff={s-a:.2f}')

## 5. Tokenizer Fragmentation Analysis

In [ ]:
print('Tokenizer features:')
print(tok_df.describe().round(3))

# Correlation: fragmentation vs bias score
corr_en, p_en = stats.pearsonr(tok_df['frag_ratio_en'].dropna(), tok_df['BiasEnglish'].dropna())
corr_fa, p_fa = stats.pearsonr(tok_df['frag_ratio_fa'].dropna(), tok_df['BiasPersian'].dropna())
print(f'\nEnglish fragmentation vs BiasScore: r={corr_en:.3f}, p={p_en:.4f}')
print(f'Persian fragmentation vs BiasScore: r={corr_fa:.3f}, p={p_fa:.4f}')

In [ ]:
# Scatter plot: fragmentation vs bias
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].scatter(tok_df['frag_ratio_en'], tok_df['BiasEnglish'], alpha=0.4, color='steelblue')
axes[0].set_title('English: Fragmentation vs BiasScore')
axes[0].set_xlabel('Fragmentation Ratio')
axes[0].set_ylabel('BiasScore')
axes[1].scatter(tok_df['frag_ratio_fa'], tok_df['BiasPersian'], alpha=0.4, color='coral')
axes[1].set_title('Persian: Fragmentation vs BiasScore')
axes[1].set_xlabel('Fragmentation Ratio')
axes[1].set_ylabel('BiasScore')
plt.tight_layout()
plt.savefig('fragmentation_vs_bias.png', dpi=150)
plt.show()

## 6. Summary Table: All Metrics

In [ ]:
summary = pd.DataFrame({
 'Bias Type': ['gender', 'race', 'religion', 'profession'],
 'LLaMA BiasScore (PLL)': Biasen[Biasen['labelname']=='stereotype'].groupby('biastype')['BiasEnglish'].mean().values.round(3),
 'PersianLLaMA BiasScore (PLL)': BiasPe[BiasPe['labelname']=='stereotype'].groupby('biastype')['BiasPersian'].mean().values.round(3),
 'LLaMA CB': cb_compare['LLaMA'].values.round(3),
 'PersianLLaMA CB': cb_compare['PersianLLaMA'].values.round(3),
})
print(summary.to_string(index=False))